# Bronze - Referências ANAC
- Aeródromos públicos
- Empresas aéreas (nacionais + estrangeiras)
- Nada de tipagem (tudo string)
- Nada de filtro
- Colunas de auditoria
- Idempotente

In [0]:
from pyspark.sql import functions as F

# Caminhos dos arquivos
CAMINHO_AERODROMOS = "/Volumes/voebem/bronze/arquivos/referencias/AerodromosPublicos.csv"
CAMINHO_EMPRESAS = "/Volumes/voebem/bronze/arquivos/referencias/pda_empresas_aereas_*.csv"

# Tabelas de destino
TABELA_AERODROMOS = "voebem.bronze.aerodromos"
TABELA_EMPRESAS = "voebem.bronze.empresas_aereas"

## 1. Aeródromos Públicos
---

In [0]:
aerodromos_bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)     # descarta linha de atualização
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")  # bronze descarta linha nenhuma
    .load(CAMINHO_AERODROMOS)
)

print("Colunas lidas dos arquivos de aeródromos:")
for c in aerodromos_bruto.columns:
    print(f" {c!r}")

### Tirando espaços e caracteres especiais das colunas

In [0]:
# Usando os nomes das colunas diretamente do DataFrame para evitar problemas de encoding
NOMES_NOVOS_AERODROMOS = [
    'codigo_oaci',
    'ciad',
    'nome',
    'municipio',
    'uf',
    'municipio_servido',
    'uf_servido',
    'lat_geopoint',
    'long_geopoint',
    'latitude',
    'longitude',
    'altitude',
    'operacao_diurna',
    'operacao_noturna',
    'situacao',
    'validade_registro',
    'portaria_registro',
    'link_portaria'
]

aerodromos_renomeado = aerodromos_bruto.select(
    *[F.col(aerodromos_bruto.columns[i]).cast("string").alias(NOMES_NOVOS_AERODROMOS[i]) 
      for i in range(len(aerodromos_bruto.columns))]
)

# Força verificação de schema
aerodromos_renomeado.printSchema()

### Colunas de auditoria

In [0]:
aerodromos_bronze = aerodromos_renomeado.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerindo_em", F.current_timestamp()
)

### Escrita idempotente

In [0]:
(
    aerodromos_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_AERODROMOS)
)

print(f"{TABELA_AERODROMOS}: {spark.table(TABELA_AERODROMOS).count():,} linhas")

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {TABELA_AERODROMOS} IS
    'Bronze - Aeródromos Públicos da ANAC.
      Dado bruto: Todas as colunas string, nenhuma linha descartada.
      Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/referencias/.'
""")

In [0]:
display(
    spark.sql(f"""
      SELECT _arquivo_origem, COUNT(*) AS linhas, MAX(_ingerindo_em) AS ingerido_em
      FROM {TABELA_AERODROMOS}
      GROUP BY _arquivo_origem
      ORDER BY _arquivo_origem
    """)
)

## 2. Empresas Aéreas (Nacionais + Estrangeiras)
---

In [0]:
empresas_bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)     # descarta linha de atualização
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")  # bronze descarta linha nenhuma
    .load(CAMINHO_EMPRESAS)
)

print("Colunas lidas dos arquivos de empresas aéreas:")
for c in empresas_bruto.columns:
    print(f" {c!r}")

### Tirando espaços e caracteres especiais das colunas

In [0]:
# Usando os nomes das colunas diretamente do DataFrame para evitar problemas de encoding
NOMES_NOVOS_EMPRESAS = [
    'icao',
    'estrangeira',
    'razao_social',
    'servico',
    'endereco',
    'cidade',
    'uf',
    'cep',
    'telefone',
    'email',
    'instrumento',
    'data',
    'ativa'
]

empresas_renomeado = empresas_bruto.select(
    *[F.col(empresas_bruto.columns[i]).cast("string").alias(NOMES_NOVOS_EMPRESAS[i]) 
      for i in range(len(empresas_bruto.columns))]
)

# Força verificação de schema
empresas_renomeado.printSchema()

### Colunas de auditoria

In [0]:
empresas_bronze = empresas_renomeado.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerindo_em", F.current_timestamp()
)

### Escrita idempotente

In [0]:
(
    empresas_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_EMPRESAS)
)

print(f"{TABELA_EMPRESAS}: {spark.table(TABELA_EMPRESAS).count():,} linhas")

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {TABELA_EMPRESAS} IS
    'Bronze - Empresas Aéreas (Nacionais + Estrangeiras) da ANAC.
      Dado bruto: Todas as colunas string, nenhuma linha descartada.
      Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/referencias/.'
""")

In [0]:
display(
    spark.sql(f"""
      SELECT _arquivo_origem, COUNT(*) AS linhas, MAX(_ingerindo_em) AS ingerido_em
      FROM {TABELA_EMPRESAS}
      GROUP BY _arquivo_origem
      ORDER BY _arquivo_origem
    """)
)